# 02 SparkSession

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand what a SparkSession is, what it gives you, and how it works in Databricks.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
SparkSession is the first thing every Spark program creates, and "what is SparkSession?" is a standard interview question. Knowing what hangs off it makes every later lesson easier.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What is SparkSession?</b><br>
<b>SparkSession</b> is the single entry point to Spark functionality. Through it you can:
<ul><li>Create DataFrames (<code>spark.createDataFrame</code>)</li><li>Read data (<code>spark.read</code>)</li><li>Run SQL (<code>spark.sql</code>)</li><li>Read and set configuration (<code>spark.conf</code>)</li><li>Access the catalog of tables and databases (<code>spark.catalog</code>)</li></ul>
In Databricks, a SparkSession is already created and available as <code>spark</code>. In your own scripts outside Databricks, you create it yourself.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A nightly ETL job starts, creates one SparkSession, reads raw files with <code>spark.read</code>, transforms them, and writes tables. Everything in the job goes through that one object.
</div>

## SparkSession vs SparkContext

| Name | Role |
|---|---|
| SparkContext | The older entry point (Spark 1.x). Connects to the cluster and creates RDDs |
| SQLContext / HiveContext | Older entry points for DataFrames and SQL |
| SparkSession | Unified entry point since Spark 2.0. Wraps all of the above |

SparkSession replaced SQLContext and HiveContext, and it holds a SparkContext inside it (on classic compute).

## Creating a SparkSession outside Databricks

```python
# Import the SparkSession class
from pyspark.sql import SparkSession

# Build a session with an application name.
# getOrCreate() reuses a running session or creates a new one.
spark = (
    SparkSession.builder
    .appName("MyApp")
    .getOrCreate()
)
```

In Databricks, `getOrCreate()` simply returns the session that already exists, so it is safe to run.

## 1. Inspect the session

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the <code>spark</code> object, the Spark version and the object type.<br><br>
<b>Why it matters</b><br>Confirms the session exists and tells you which Spark version you are using.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A SparkSession description, a version string, and a type that mentions <code>SparkSession</code>. On serverless the type is the Spark Connect version: <code>pyspark.sql.connect.session.SparkSession</code>.
</div>

In [0]:
# 'spark' is the SparkSession that Databricks creates for us automatically.
# Printing it shows that the session object exists.
print(spark)

# spark.version tells us which Apache Spark version is running.
print("Spark version:", spark.version)

# type() shows what kind of object 'spark' is (it should mention SparkSession).
print("Type:", type(spark))

## 2. getOrCreate in Databricks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Asks for a session using the builder and compares it to <code>spark</code>.<br><br>
<b>Why it matters</b><br>Shows that <code>getOrCreate()</code> reuses the existing session instead of making a new one.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>True</code> or <code>False</code>. On serverless the result can differ from classic compute, so write down what you see.
</div>

In [0]:
# Import the SparkSession class so we can use its builder.
from pyspark.sql import SparkSession

# getOrCreate() returns the session that is already running.
# If none existed, it would create a new one.
same_spark = SparkSession.builder.getOrCreate()

# Check whether we got back the very same object as 'spark'.
print("Same session as spark?", same_spark is spark)

## 3. What you can do with SparkSession

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a DataFrame from a Python list, generates numbers with <code>range</code>, and runs a SQL query.<br><br>
<b>Why it matters</b><br>These three calls (<code>createDataFrame</code>, <code>range</code>, <code>sql</code>) are the most common ways to start working with data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 4-row table with salaries, a 5-row table of ids 0 to 4, and a one-row table with <code>one</code> and <code>greeting</code>.
</div>

In [0]:
# 1. Create a DataFrame from a Python list.
#    Each tuple is one row of data.
data = [
    (1, "Amal", "India", 21000),
    (2, "Senthil", "USA", 150000),
    (3, "John", "UK", 18000),
    (4, "Vivek", "UAE", 25000),
]

# The column names, in the same order as the values in each tuple.
columns = ["id", "name", "country", "salary"]

# createDataFrame() builds a distributed table from the data and column names.
df = spark.createDataFrame(data, columns)

# show() prints the DataFrame as a plain text table.
df.show()

# 2. spark.range(5) generates a DataFrame with one column 'id' holding 0 to 4.
#    It is handy for quick tests when you have no data.
spark.range(5).show()

# 3. spark.sql() runs a SQL query and returns the result as a DataFrame.
#    Here we select two literal values, so no table is needed.
spark.sql("SELECT 1 AS one, 'hello' AS greeting").show()

## 4. Catalog and configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the current catalog and schema, then tries to read one Spark setting.<br><br>
<b>Why it matters</b><br>The catalog tells you where tables are created by default. Configs control how Spark runs, but serverless restricts many of them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your current catalog and database (often <code>workspace</code> and <code>default</code>), and a shuffle partitions value or a "not readable" message.
</div>

In [0]:
# The catalog is the list of data objects Spark can see (catalogs, schemas, tables).
# try/except keeps the notebook running if a call isn't supported on this compute.
try:
    # The catalog we are currently working in.
    print("Current catalog:", spark.catalog.currentCatalog())
    # The schema (database) we are currently working in.
    print("Current database:", spark.catalog.currentDatabase())
except Exception as e:
    print("Not available here:", e)

# spark.conf reads Spark settings.
# 'shuffle partitions' controls how many pieces data is split into after a shuffle.
# Serverless may not let us read it, so we handle that case.
try:
    print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception as e:
    print("Config not readable here:", e)

## 5. SparkContext on serverless

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to read the application name through <code>spark.sparkContext</code>.<br><br>
<b>Why it matters</b><br>SparkContext is the older entry point inside SparkSession. On serverless it is restricted, which is an important difference from classic compute.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Most likely the "not available" message. Write down what you see.
</div>

In [0]:
# SparkContext is the older, lower-level entry point that SparkSession wraps.
# On classic compute we can reach it with spark.sparkContext.
# On serverless this is restricted, so we expect an error and we catch it.
try:
    print("App name:", spark.sparkContext.appName)
except Exception as e:
    # type(e).__name__ prints just the error type, keeping the output short.
    print("sparkContext is not available on serverless:", type(e).__name__)

## What Spark is doing internally

```
Your Python code (driver)
        |
        v
   SparkSession   <-- entry point
        |
        v
  Builds query plans
        |
        v
  Cluster runs the work on executors
        |
        v
  Results return to the driver
```

The SparkSession lives in the **driver**. It builds the plan, and the cluster executes it. One session per application is the normal pattern.

**On serverless (Spark Connect):** the `spark` in your notebook is a thin **client**. It sends the plan over the network to a Spark server that Databricks runs for you, and receives the results. That's why `spark.sparkContext` isn't available there.

## Deploy modes: where does the driver run?

When you submit a Spark application, you choose where the **driver** runs. This is called the **deploy mode**.

| Mode | Where the driver runs | Typical use |
|---|---|---|
| Local | Everything in one process on one machine | Learning, unit tests |
| Client | On the machine that submits the application. Executors run on the cluster | Interactive work (notebooks, spark-shell), debugging |
| Cluster | Inside the cluster, on a node chosen by the cluster manager. Executors also run on the cluster | Production batch jobs |

```
CLIENT MODE                              CLUSTER MODE

Your machine                             Your machine
+----------------+                       +----------------+
|    Driver      |                       |  spark-submit  |  (can disconnect)
+-------+--------+                       +-------+--------+
        |                                        |
        v                                        v
Cluster                                  Cluster
+----+ +----+ +----+                     +--------+ +----+ +----+
| E1 | | E2 | | E3 |                     | Driver | | E1 | | E2 |
+----+ +----+ +----+                     +--------+ +----+ +----+
```

## Client vs cluster: trade-offs

| | Client mode | Cluster mode |
|---|---|---|
| Output and logs | Appear directly on your machine | Collected from the cluster |
| If your machine disconnects | The application fails, because the driver is gone | The application keeps running |
| Network | Driver talks to executors across a network, which can be slow | Driver sits next to the executors |
| Best for | Development and interactive sessions | Production and long-running jobs |

## How you choose it

Outside Databricks, you pass it to `spark-submit`. This is a terminal command, not a notebook cell:

```
# Client mode on YARN
spark-submit --master yarn --deploy-mode client my_job.py

# Cluster mode on YARN
spark-submit --master yarn --deploy-mode cluster my_job.py
```

## Cluster managers (the `--master` value)

| Manager | Notes |
|---|---|
| Local (`local[*]`) | One machine, uses all its cores |
| Standalone (`spark://host:7077`) | Spark's own simple manager. Per the Spark docs, cluster mode is not supported for Python applications here |
| YARN | Common in Hadoop environments |
| Kubernetes (`k8s://...`) | Runs driver and executors as pods |
| Mesos | Deprecated, avoid for new work |

## What about Databricks?

In Databricks you don't pick a deploy mode. The notebook is attached to compute, and the driver runs on the driver node **inside** that compute, with the executors on worker nodes. That is conceptually closest to cluster mode. Your browser is only a window, so closing it doesn't stop a running job. On serverless compute you don't see the driver and executors at all.

## 6. Check the master and deploy mode

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to read the master URL and deploy mode from the Spark settings.<br><br>
<b>Why it matters</b><br>Outside Databricks these tell you where the application runs. On Databricks and serverless they are often hidden or restricted.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A value or a "not readable" message. Both are useful observations, so write down what you see.
</div>

In [0]:
# spark.master is the cluster manager URL (for example 'local[*]' or 'yarn').
# spark.submit.deployMode says whether the driver is in 'client' or 'cluster' mode.
# Databricks and serverless may hide these, so each read is wrapped in try/except.
for key in ["spark.master", "spark.submit.deployMode"]:
    try:
        # spark.conf.get() reads a Spark setting by name.
        print(key, "=", spark.conf.get(key))
    except Exception as e:
        # type(e).__name__ prints just the error type, keeping the output short.
        print(key, "is not readable here:", type(e).__name__)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>sparkContext</code> raises an error on serverless</b><br>
Expected, serverless restricts it. Use <code>spark</code> APIs instead.<br><br>
**⛔ Problem: <code>spark.conf.set(...)</code> rejected**<br>Some configs can't be changed on serverless. Note which ones.<br><br>
**⛔ Problem: <code>NameError: spark is not defined</code>**<br>Attach the notebook to compute and re-run.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is SparkSession?</b><br>
The unified entry point to Spark. It is used to create DataFrames, read data, run SQL and manage configuration.<br><br>

<b>🎤 2. What is the difference between SparkSession and SparkContext?</b><br>
SparkContext is the older low-level entry point that connects to the cluster and creates RDDs. SparkSession, introduced in Spark 2.0, wraps SparkContext, SQLContext and HiveContext into one object.<br><br>

<b>🎤 3. What does <code>getOrCreate()</code> do?</b><br>
Returns the existing SparkSession if there is one, otherwise creates a new one.<br><br>

<b>🎤 4. Do you need to create a SparkSession in Databricks?</b><br>
No, a session called <code>spark</code> is created for you.<br><br>

<b>🎤 5. Can you have multiple SparkSessions?</b><br>
Yes. They can share one SparkContext but have separate SQL configs and temp views. In practice most applications use one.<br><br>

<b>🎤 6. Where does the SparkSession run?</b><br>
In the driver. The executors only run tasks.<br><br>

<b>🎤 7. What is the difference between client mode and cluster mode?</b><br>
In client mode the driver runs on the machine that submits the application. In cluster mode the driver runs inside the cluster, on a node chosen by the cluster manager. Executors run on the cluster in both modes.<br><br>

<b>🎤 8. Which mode would you use for production and why?</b><br>
Cluster mode. The driver sits next to the executors, so there is less network latency, and the job keeps running even if the submitting machine disconnects.<br><br>

<b>🎤 9. Which mode is used for interactive work like spark-shell or notebooks?</b><br>
Client mode, because the driver must be reachable so you can see results and type commands.<br><br>

<b>🎤 10. What happens in client mode if the submitting machine goes down?</b><br>
The driver dies with it, so the application fails.<br><br>

<b>🎤 11. What is local mode?</b><br>
The driver and executors run in one process on one machine. It's used for learning and testing, not for real workloads.<br><br>

<b>🎤 12. What deploy mode does Databricks use?</b><br>
You don't choose one. The driver runs on the driver node inside the compute, which behaves like cluster mode.<br><br>

<b>🎤 13. What is Spark Connect?</b><br>
A client-server architecture introduced in Spark 3.4. The client (for example a serverless notebook) builds DataFrame plans and sends them to a remote Spark server, which executes them. The client has no <code>sparkContext</code> or RDD access.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In a Databricks notebook, how do you get the SparkSession?</b><br>
A. Create it with <code>SparkSession.builder.master("local").getOrCreate()</code><br>
B. It is already available as the variable <code>spark</code><br>
C. Import it from <code>dbutils</code><br>
D. Call <code>SparkContext.getOrCreate()</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Databricks creates it for every notebook.</details><br><br>

<b>Q2. Which method on <code>spark</code> runs a SQL statement and returns a DataFrame?</b><br>
A. <code>spark.query()</code><br>
B. <code>spark.execute()</code><br>
C. <code>spark.sql()</code><br>
D. <code>spark.read.sql()</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Without copying from above:
<ol><li>Write the code to create a SparkSession named <code>PracticeApp</code> using the builder pattern (as a script you would run outside Databricks)</li><li>Use <code>spark</code> to create a DataFrame of 3 cities with columns <code>id</code> and <code>city</code></li><li>Use <code>spark.sql</code> to return today's date with <code>SELECT current_date()</code></li><li>Print the current catalog and database</li><li>In one sentence: what is the difference between SparkSession and SparkContext?</li></ol>
</div>

In [0]:
# Build and practice here

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Outside Databricks (shown as text, because Databricks already has a session):
#    from pyspark.sql import SparkSession
#    spark = SparkSession.builder.appName("PracticeApp").getOrCreate()

# 2. A small DataFrame
cities = spark.createDataFrame([(1, "Chennai"), (2, "Dubai"), (3, "London")], ["id", "city"])
cities.show()

# 3. Today's date with SQL
spark.sql("SELECT current_date() AS today").show()

# 4. Current catalog and database
print(spark.catalog.currentCatalog(), spark.catalog.currentDatabase())

# 5. SparkContext is the old low-level entry point for RDDs. SparkSession is the unified
#    entry point for DataFrames, SQL, config and catalog, and wraps SparkContext.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>SparkSession is the single entry point to Spark</li><li>It gives you <code>createDataFrame</code>, <code>read</code>, <code>sql</code>, <code>conf</code> and <code>catalog</code></li><li>Databricks creates it for you as <code>spark</code></li><li><code>getOrCreate()</code> reuses an existing session</li><li>SparkSession replaced SQLContext and HiveContext, and wraps SparkContext</li><li>On serverless (Spark Connect), <code>sparkContext</code> access is restricted</li><li>Deploy modes: local, client (driver on your machine), cluster (driver inside the cluster)</li></ul>
</div>

| Task | Code |
|---|---|
| Version | `spark.version` |
| Create a DataFrame | `spark.createDataFrame(data, cols)` |
| Numbers | `spark.range(5)` |
| SQL | `spark.sql("SELECT ...")` |
| Config | `spark.conf.get(key)`, `spark.conf.set(key, value)` |
| Catalog | `spark.catalog.currentCatalog()`, `currentDatabase()` |
| Outside Databricks | `SparkSession.builder.appName("x").getOrCreate()` |

## Git commit

```
git add 01_spark_basics/02_spark_session.ipynb
git commit -m "restyle 02 spark session to lesson standard"
```